# 第 15 章　分析の統合

問い・測定・日付の対応・比較条件を一つの分析につなぎ，気象の記事を書きます。

このノートの前半は，本文の説明と一緒に実行します。各場面で先に短い問いを読み，講師の案内に合わせてセルを動かし，出力の見所を確認します。後半は約20分の実習です。前半のコードは動く見本として用意してあり，作図コードをすべて書き写す必要はありません。

Colab では「ドライブにコピーを保存」で自分用のノートを作ります。CSV は，手元にファイルがなければ公開 URL から読み込みます。ローカルでは，配布 ZIP の配置のまま開くか，ノートと CSV を同じフォルダーに置いて実行できます。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

file_name = "jma-temperature-2024-08.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

temperatures = pd.read_csv(
    csv_path, dtype={"station_code": str}, parse_dates=["date"],
)
station_codes = ["47626", "47807"]
station_labels = ["Kumagaya", "Fukuoka"]
station_colors = ["#195b8a", "#a7421f"]

temperatures[["station", "date", "max_temp_c", "min_temp_c"]].head()

## 本文と一緒に実行

### 1. 「暑かった」を測定へ置き換える

8月7日の二地点では，日最高気温と日最低気温のどちらを比べるかで大小が入れ替わります。原表で地点・日付・項目を確認してから，三つの気温を描きます。

In [ ]:
example_date = pd.Timestamp("2024-08-07")
temperature_fields = ["mean_temp_c", "max_temp_c", "min_temp_c"]
one_date = temperatures.loc[temperatures["date"] == example_date]
one_date[["station", "date"] + temperature_fields]

In [ ]:
plt.rcParams["figure.figsize"] = (7.2, 2.7)
fig, ax = plt.subplots(layout="constrained")
for level, code, color in zip([1, 0], station_codes, station_colors):
    day = one_date.loc[one_date["station_code"] == code].iloc[0]
    low, mean, high = day["min_temp_c"], day["mean_temp_c"], day["max_temp_c"]
    ax.hlines(level, low, high, color=color, linewidth=3)
    ax.scatter([low, high], [level, level], color=color, marker="|", s=190)
    ax.scatter(mean, level, color=color, marker="D", s=41, zorder=3)
    for value, name in [(low, "Min"), (mean, "Mean"), (high, "Max")]:
        ax.annotate(f"{name} {value:.1f}", (value, level), xytext=(0, 11),
                    textcoords="offset points", ha="center", color=color, fontsize=9)
ax.set(xlabel="Temperature (°C)", yticks=[1, 0], yticklabels=station_labels,
       xlim=(22.5, 37.5), ylim=(-0.4, 1.6))
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(axis="y", length=0)
plt.show()

**出力の見所：**線の左右の端は日最低・日最高気温，ひし形は日平均です。日平均は両端の中点でしょうか。日最低気温は一日全体の最低で，夜間の最低気温を直接集計した列ではありません。

### 2. 分母を確かめて日数を数える

一地点31行であれば，全31日が一回ずつそろうと言えるでしょうか。地点と日付の重複，値の欠損，暦との対応を確認します。

In [ ]:
station_day = ["station_code", "date"]
by_station = temperatures.groupby(["station_code", "station"])
coverage = by_station.agg(
    rows=("date", "size"),
    dates=("date", "nunique"),
    maximum_values=("max_temp_c", "count"),
    minimum_values=("min_temp_c", "count"),
    mean_values=("mean_temp_c", "count"),
)
print("Duplicate station-date rows:", temperatures.duplicated(station_day).sum())
coverage

In [ ]:
august = pd.date_range("2024-08-01", "2024-08-31", freq="D")
for (code, station), station_rows in by_station:
    dates = pd.DatetimeIndex(station_rows["date"]).sort_values()
    print(code, station, dates.equals(august))

ordered_temperatures = (
    (temperatures["min_temp_c"] <= temperatures["mean_temp_c"])
    & (temperatures["mean_temp_c"] <= temperatures["max_temp_c"])
)
print("Rows with minimum ≤ mean ≤ maximum:", ordered_temperatures.sum())

In [ ]:
criteria = temperatures.assign(
    maximum_at_least_35=temperatures["max_temp_c"] >= 35,
    minimum_at_least_25=temperatures["min_temp_c"] >= 25,
)
criteria[["station", "date", "max_temp_c", "maximum_at_least_35"]].loc[
    criteria["max_temp_c"] == 35
]

In [ ]:
monthly_counts = criteria.groupby("station_code").agg(
    days=("date", "size"),
    days_max_at_least_35=("maximum_at_least_35", "sum"),
    days_min_at_least_25=("minimum_at_least_25", "sum"),
)
monthly_counts["percent_max_at_least_35"] = (
    100 * monthly_counts["days_max_at_least_35"] / monthly_counts["days"]
)
monthly_counts["percent_min_at_least_25"] = (
    100 * monthly_counts["days_min_at_least_25"] / monthly_counts["days"]
)
monthly_counts.round(1)

**出力の見所：**日最高35℃以上は熊谷20日・福岡17日，日最低25℃以上は熊谷24日・福岡30日です。35.0℃の日を含める条件はどの記号で表されていますか。

### 3. 同じ日を対応させる

月の日数だけでは見えない日ごとの差を確かめます。日付順に描いた後，日付をキーに結合します。図の一点は何を一組にした値か，説明してください。

In [ ]:
import matplotlib.dates as mdates
kuma_code, fuku_code = station_codes
rules = [("max_temp_c", 35, "Daily maximum ≥ 35 °C"),
         ("min_temp_c", 25, "Daily minimum ≥ 25 °C")]
plt.rcParams["figure.figsize"] = (7.2, 5.4)
fig, axes = plt.subplots(2, 1, sharex=True, sharey=True, layout="constrained")
for ax, (measure, threshold, title) in zip(axes, rules):
    for code, label, color, style in zip(
        station_codes, station_labels, station_colors, ["-", "--"],
    ):
        station_rows = temperatures.loc[temperatures["station_code"] == code]
        station_rows = station_rows.sort_values("date")
        ax.plot(station_rows["date"], station_rows[measure], label=label,
                color=color, linestyle=style, marker=".", markersize=3.6)
    ax.axhline(threshold, color="#6e7880", linestyle=":", linewidth=1.1)
    ax.set(ylabel="Temperature (°C)", title=title.split(" ≥ ")[0], ylim=(22, 40))
axes[0].legend(frameon=False, ncol=2, loc="lower left")
axes[-1].set_xticks(august[::5])
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
axes[-1].set_xlabel("Date (2024)")
plt.show()

In [ ]:
pair_fields = ["date", "max_temp_c", "min_temp_c"]
kumagaya = temperatures.loc[temperatures["station_code"] == kuma_code, pair_fields]
fukuoka = temperatures.loc[temperatures["station_code"] == fuku_code, pair_fields]

paired = kumagaya.merge(
    fukuoka,
    on="date",
    how="outer",
    validate="one_to_one",
    suffixes=("_kumagaya", "_fukuoka"),
    indicator=True,
).sort_values("date")
paired["_merge"].value_counts()

In [ ]:
paired["maximum_gap_c"] = (
    paired["max_temp_c_kumagaya"] - paired["max_temp_c_fukuoka"]
).round(1)
paired["minimum_gap_c"] = (
    paired["min_temp_c_kumagaya"] - paired["min_temp_c_fukuoka"]
).round(1)
paired.loc[paired["date"] == example_date, ["date", "maximum_gap_c", "minimum_gap_c"]]

In [ ]:
plt.rcParams["figure.figsize"] = (6.5, 4.8)
fig, ax = plt.subplots(layout="constrained")
ax.axvline(0, color="#77838c", linestyle=":", linewidth=1)
ax.axhline(0, color="#77838c", linestyle=":", linewidth=1)
ax.scatter(paired["maximum_gap_c"], paired["minimum_gap_c"], s=30, color="#465f72")
for day_number, offset in [(7, (-67, -16)), (12, (-28, 8)), (17, (-35, 17)), (22, (9, 3))]:
    row = paired.loc[paired["date"].dt.day == day_number].iloc[0]
    coords = (row["maximum_gap_c"], row["minimum_gap_c"])
    ax.scatter(*coords, s=47, facecolor="white", edgecolor=station_colors[0], zorder=3)
    ax.annotate(f"Aug {day_number}", coords, xytext=offset,
                textcoords="offset points", fontsize=9,
                arrowprops={"arrowstyle": "-", "color": "#697a88", "linewidth": 0.7})
ax.set(xlabel="Daily maximum: Kumagaya − Fukuoka (°C)",
       ylabel="Daily minimum: Kumagaya − Fukuoka (°C)",
       xlim=(-6, 6), ylim=(-5.7, 3), xticks=[-6, -4, -2, 0, 2, 4, 6])
plt.show()

In [ ]:
gap_counts = pd.DataFrame({
    "Kumagaya higher": (paired[["maximum_gap_c", "minimum_gap_c"]] > 0).sum(),
    "Equal": (paired[["maximum_gap_c", "minimum_gap_c"]] == 0).sum(),
    "Fukuoka higher": (paired[["maximum_gap_c", "minimum_gap_c"]] < 0).sum(),
})
gap_counts

**出力の見所：**右下の点は日最高気温が熊谷で高く，日最低気温が福岡で高い日です。日最低気温が福岡で高い28日という数は，25℃以上の30日という数と何が違うでしょうか。

### 4. 二地点の該当日は重なるか

熊谷20日と福岡17日を足すと，「少なくとも片方で35℃以上だった暦日の日数」になるでしょうか。四通りのTrue/Falseを数えます。

In [ ]:
kuma_reached = paired["max_temp_c_kumagaya"] >= 35
fuku_reached = paired["max_temp_c_fukuoka"] >= 35
pd.crosstab(
    kuma_reached.rename("Kumagaya"),
    fuku_reached.rename("Fukuoka"),
)

**出力の見所：**両地点で該当した日を二回数えていないか確認します。表の各セルから，地点別の20日と17日を作り直してください。

### 5. 基準だけを動かす

猛暑日の定義は35℃以上です。別の問いとして34℃以上・36℃以上も調べると，どちらが多いかは同じでしょうか。期間は8月全体のままにして関数へ渡します。

In [ ]:
def count_days_at_or_above(data, temperature_column, threshold):
    """Count qualifying days for each station."""
    values = data[temperature_column]
    reached = values >= threshold
    return reached.groupby(data["station_code"]).sum()


thresholds = [34, 35, 36]
counts_by_threshold = pd.DataFrame(
    [count_days_at_or_above(temperatures, "max_temp_c", limit) for limit in thresholds],
    index=pd.Index(thresholds, name="threshold_c"),
)
counts_by_threshold

In [ ]:
plt.rcParams["figure.figsize"] = (6.5, 3.7)
fig, ax = plt.subplots(layout="constrained")
for code, label, color, marker, style in zip(
    station_codes, station_labels, station_colors, ["s", "o"], ["-", "--"],
):
    ax.plot(counts_by_threshold.index, counts_by_threshold[code], marker=marker,
            linestyle=style, color=color, label=label)
ax.set(xlabel="Daily maximum temperature threshold (°C)", ylabel="Days at or above",
       xticks=thresholds, ylim=(0, 31), xlim=(33.85, 36.15))
ax.legend(frameon=False)
plt.show()

In [ ]:
monthly_maximum = by_station["max_temp_c"].agg(["mean", "median", "max"])
monthly_maximum.round(2)

**出力の見所：**34℃から35℃に基準を上げたとき，どの温度範囲の日が数える対象から外れますか。平均・中央値・最大が答える問いとも比べ，記事で最初に決めた問いに合う数を選びます。

### 6. 図から一文を書く

講義で読んだ数を一つ選び，地点・期間・測定項目・分母が分かる一文を書いてください。別の年にも同じ傾向があると述べるには，何を追加で調べるかも一つ挙げます。

（ここに一文と次に調べることを書く）

## Jupyter 実習（約20分）

ここからは8月全体から期間を替え，前半と後半を比べます。基準の35℃・25℃は保ちます。以下のセルで期間を指定し，講義の月全体の値をそのまま解答に使わず，選んだ日数から報告します。

## 1. 月の前半を読む（約 6 分）

日付の範囲を確認してから，二つの気温の条件で日数を数えます。1 行が「一地点の一日」を表すことを確認し，`days` が何日分かを見てください。基準ちょうどの日も数えます。

次の二つのセルで表と図を出します。二つの条件について，月の前半の熊谷・福岡の該当日数と分母を記録します。

In [ ]:
first_day = 1
last_day = 15

start = pd.Timestamp(year=2024, month=8, day=first_day)
end = pd.Timestamp(year=2024, month=8, day=last_day)
period = temperatures.loc[temperatures["date"].between(start, end)].copy()
period["maximum_at_least_35"] = period["max_temp_c"] >= 35
period["minimum_at_least_25"] = period["min_temp_c"] >= 25
summary = period.groupby("station_code").agg(
    days=("date", "size"),
    maximum_days=("maximum_at_least_35", "sum"),
    minimum_days=("minimum_at_least_25", "sum"),
)
summary = summary.reindex(station_codes)
summary

In [ ]:
rules = [("maximum_days", "Daily maximum ≥ 35 °C"),
         ("minimum_days", "Daily minimum ≥ 25 °C")]
fig, axes = plt.subplots(1, 2, sharex=True, sharey=True, figsize=(7.2, 3.0),
                         layout="constrained")
for ax, (count_column, title) in zip(axes, rules):
    shares = 100 * summary[count_column] / summary["days"]
    bars = ax.barh(station_labels, shares, color=station_colors)
    ax.bar_label(bars, labels=[f"{n}/{d}" for n, d in
                               zip(summary[count_column], summary["days"])], padding=4)
    ax.set(title=title, xlabel="Share of selected days (%)", xlim=(0, 116),
           xticks=[0, 25, 50, 75, 100])
axes[0].invert_yaxis()
fig.suptitle(f"August {first_day}–{last_day}, 2024")
plt.show()

（ここに月の前半の該当日数と分母を書く）

## 2. 比べる期間だけを変える（約 7 分）

後半でも，日最高気温が 35 ℃以上の日が多い地点は同じだと思いますか。前半の図を根拠に，一文で予想します。

次に `first_day` と `last_day` を変更して，8 月 16〜31 日を選びます。集計・作図のセルを再実行し，`days` が両地点とも何日になったかを確認します。二つの条件について，それぞれの該当日数と分母を記録してください。

（ここに予想と月の後半の該当日数・分母を書く）

## 3. 記事の説明を書き換える（約 7 分）

「熊谷の方が暑かった」という記事の一文を，この前半・後半の記録に基づく 3〜4 文へ書き換えます。次の内容を文章に含めます。

- どの観測地点の，いつの気温を比べたか。
- 「日最高気温 35 ℃以上の日」「日最低気温 25 ℃以上の日」について，前半と後半で地点差がどう変わったか。該当日数と分母を使う。
- この月の結果を他の年についても述べるなら，どの観測を追加するか。

（ここに書き換えた説明を書く）

気象庁の「熱帯夜」は夜間の最低気温が 25 ℃以上の場合を指します。このノートブックは日別表の日最低気温を数えているので，記事でも「日最低気温が 25 ℃以上の日」と書きます。 [気象庁の気温の用語](https://www.jma.go.jp/jma/kishou/know/yougo_hp/kion.html)

提出する場合はカーネルを再起動して上から再実行し，後半の図と両期間の数値を含む説明を保存します。

### 出典

気象庁「過去の気象データ検索」の日ごとの値。熊谷と福岡の 2024 年 8 月の観測値から気温列を教材用に抽出・加工した CSV を使用しました。 [熊谷の原表](https://www.data.jma.go.jp/stats/etrn/view/daily_s1.php?prec_no=43&block_no=47626&year=2024&month=8&day=&view=p1) ・ [福岡の原表](https://www.data.jma.go.jp/stats/etrn/view/daily_s1.php?prec_no=82&block_no=47807&year=2024&month=8&day=&view=p1) ・ [気象庁の利用規約](https://www.jma.go.jp/jma/kishou/info/coment.html)